# 02 — Stabl Feature Selection

**Pipeline Step 2 of 5**

This notebook applies **Stabl** (Stability Selection with L1-penalised Logistic Regression) to objectively reduce the transcriptomic feature space from thousands of highly variable genes down to a compact set of the most stable biomarker candidates.

### Why Stabl?

Traditional gene selection methods (e.g., fold-change thresholds, p-value cutoffs) are sensitive to sample composition and often produce unstable gene lists that change substantially with small perturbations in the data. Stabl addresses this problem by repeatedly subsampling (bootstrapping) the dataset and fitting an L1-penalised model on each subsample. A gene is considered "stable" only if it is consistently selected across many bootstrap iterations. The algorithm then applies the FDP+ (False Discovery Proportion) framework to set an automatic, data-driven threshold — no manual tuning is required.

The result is a gene set with controlled false-discovery guarantees, meaning the selected genes are genuinely informative rather than artifacts of noise or batch effects. This is particularly important for downstream knowledge graph construction, where we want high-confidence nodes.

### Inputs
| File | Description |
|---|---|
| `data/processed/mouse_brain_preprocessed.h5ad` | QC-filtered, normalized AnnData from Step 01 |

### Outputs
| File | Description |
|---|---|
| `cache/stabl_results_<hash>.pkl` | Full Stabl result dictionary (stability scores, thresholds, selected genes) |
| `cache/stabl_features_<hash>.csv` | Table of selected genes with their stability scores |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.spatial_pipeline import load_adata, run_stabl_cached

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

Imports ready.


## 2.1 Load Preprocessed Data

We load the QC-filtered and normalized AnnData produced by notebook 01. At this point the data has already been filtered for low-quality spots and rarely detected genes, and expression values have been library-size normalized and log-transformed.

In [2]:
adata = load_adata(DATA_PROCESSED / "mouse_brain_preprocessed.h5ad")
print(f"\nLoaded: {adata.shape[0]} spots × {adata.shape[1]} genes")

  Loading dataset: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/processed/mouse_brain_preprocessed.h5ad
  Shape: 2691 spots × 19652 genes

Loaded: 2691 spots × 19652 genes


## 2.2 Run Stabl Feature Selection

The Stabl pipeline proceeds in four stages:

1. **Highly Variable Gene (HVG) selection (2,000 genes):** From the full gene set, we select the 2,000 most variable genes using scanpy's HVG method. This initial reduction removes genes with near-constant expression that carry no discriminative information, while keeping the feature space manageable for the bootstrap procedure.

2. **Binary label assignment (Leiden clustering):** Stabl requires a supervised binary target. Since this dataset does not have injury/control annotations, we use Leiden graph-based clustering to partition spots into two groups: cortical/hippocampal (label 1) vs. subcortical (label 0). This serves as a biologically meaningful proxy — the cortex and hippocampus have distinct transcriptional signatures from subcortical structures, providing a genuine signal for Stabl to learn.

3. **Bootstrap stability selection (500 iterations):** For each of 500 bootstrap samples, an L1-penalised logistic regression model is fitted. The frequency with which each gene is selected across all bootstraps defines its stability score (ranging from 0 to 1). Genes that appear consistently — regardless of which subset of spots is used — receive high stability scores.

4. **FDP+ thresholding:** Stabl injects synthetic random-permutation features alongside the real genes and uses the selection frequency of these synthetic features to estimate the False Discovery Proportion. The threshold is set automatically to control FDP+ at the natural data-driven level.

Results are cached to a pickle file keyed by a hash of the input parameters. On subsequent runs with the same data and settings, cached results load instantly.

In [3]:
stabl_result = run_stabl_cached(
    adata,
    cache_dir=CACHE_DIR,
    dataset_name="squidpy",
    n_hvgs=2000,
    label_method="cluster",
    n_bootstraps=500,
)

print(f"\nStabl results:")
print(f"  Features selected: {stabl_result['n_selected']}")
print(f"  FDP+ threshold: {stabl_result['threshold']:.4f}")
print(f"  Minimum FDP+: {stabl_result['fdr']:.4f}")

  Loading cached Stabl results: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/cache/stabl_results_eba9686a7ee7.pkl

Stabl results:
  Features selected: 41
  FDP+ threshold: 0.8700
  Minimum FDP+: 0.0244


## 2.3 Review Selected Features

Below we display the Stabl-certified biomarker genes ranked by their stability score. A score of 1.0 means the gene was selected in every single bootstrap iteration — these are the most robust markers. Genes with scores closer to the threshold are still statistically reliable (they passed the FDP+ test) but are somewhat less consistently selected.

The stability score distribution helps assess the overall signal quality in the data. A bimodal distribution (many genes near 0 and a clear set near 1) indicates strong, well-separated signal. A more gradual distribution suggests subtler differences between the two groups.

In the context of this mouse brain dataset, we expect to see genes that mark major brain cell types (neurons, astrocytes, oligodendrocytes) and regionally enriched transcripts, since these are the biological features that most strongly distinguish cortical from subcortical spots.

In [4]:
import pandas as pd

df_features = pd.DataFrame({
    "Gene": stabl_result["selected_genes"],
    "Stability Score": [
        stabl_result["stability_scores"][g]
        for g in stabl_result["selected_genes"]
    ],
}).sort_values("Stability Score", ascending=False).reset_index(drop=True)

print(f"\nTop Stabl-certified biomarker genes ({len(df_features)} total):")
df_features.head(20)


Top Stabl-certified biomarker genes (41 total):


,Gene,Stability Score
0,Kcnip2,1.000
1,Wnt4,1.000
2,Pcdh8,1.000
3,Cplx2,1.000
4,Stxbp6,1.000
5,Ctxn3,1.000
6,Lpl,1.000
7,Lmo3,1.000
8,Penk,1.000
9,Lamp5,1.000


In [5]:
# Distribution of stability scores
print(f"Score statistics:")
print(f"  Mean: {df_features['Stability Score'].mean():.4f}")
print(f"  Median: {df_features['Stability Score'].median():.4f}")
print(f"  Min: {df_features['Stability Score'].min():.4f}")
print(f"  Max: {df_features['Stability Score'].max():.4f}")
print(f"  Genes with score = 1.0: {(df_features['Stability Score'] == 1.0).sum()}")

Score statistics:
  Mean: 0.9550
  Median: 0.9700
  Min: 0.8720
  Max: 1.0000
  Genes with score = 1.0: 10
